# TP2 : Introduction aux Algorithmes de Machine Learning Supervisé et Non supervisé

Dans ce TP,  vous travaillerez sur le dataset Iris.
Les algorithmes que vous allez tester sont :
1. classification Naive Bayes
2. KNN
3. PCA
4. GMM

In [ ]:
from sklearn import datasets
iris = datasets.load_iris()

In [ ]:
print(type(iris))

In [ ]:
print(dir(iris))

In [ ]:
iris.feature_names

In [ ]:
iris.data[:5]

In [ ]:
iris.target_names

In [ ]:
target = iris.target # Les labels associés à chaque enregistrement
target  

In [ ]:
for i in [0,1,2]:
    print("classe : %s, nb exemplaires: %s" % (i, len(target[ target == i]) ) )


In [ ]:
print(iris.DESCR)

In [ ]:
data = iris.data 

In [ ]:
# tableau numpy de 2 dimensions de 150 enregistrements de 4 valeurs
type(data), data.ndim, data.shape 


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

fig = plt.figure(figsize=(8, 4))
fig.subplots_adjust(hspace=0.4, wspace=0.4)
ax1 = plt.subplot(1,2,1)

clist = ['violet', 'yellow', 'blue']
colors = [clist[c] for c in iris.target]

ax1.scatter(data[:, 0], data[:, 1], c=colors)
plt.xlabel('Longueur du sepal (cm)')
plt.ylabel('Largueur du sepal (cm)')

ax2 = plt.subplot(1,2,2)

ax2.scatter(data[:, 2], data[:, 3], color=colors)

plt.xlabel('Longueur du petal (cm)')
plt.ylabel('Largueur du petal (cm)')

# Légende
for ind, s in enumerate(iris.target_names):
    # on dessine de faux points, car la légende n'affiche que les points ayant un label
    plt.scatter([], [], label=s, color=clist[ind])
    
plt.legend(scatterpoints=1, frameon=False, labelspacing=1
           , bbox_to_anchor=(1.8, .5) , loc="center right", title='Espèces')
plt.plot();

In [ ]:
import seaborn as sns
import pandas as pd
sns.set()
df = pd.DataFrame(data, columns=iris['feature_names'] )
df['target'] = target
df['label'] = df.apply(lambda x: iris['target_names'][int(x.target)], axis=1)
#df=df.sample(10)
df.head()


In [ ]:
sns.pairplot(df, hue='label', vars=iris['feature_names'], height=2);


# Apprentissage

## 1. Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
#Création du classifieur
clf = GaussianNB()

In [ ]:
#Apprentissage
clf.fit(data, target) # On aurait aussi pu utiliser le dataframe df


In [ ]:
GaussianNB(priors=None)

In [ ]:
print(dir(clf))

In [ ]:
clf.get_params()

In [ ]:
#Exécutons la prédiction sur les données d'apprentissage elles-mêmes
result = clf.predict(data)

In [ ]:
result

In [ ]:
# Qualité de la prédiction
result - target

In [ ]:
#Pourcentage d'erreur
errors = sum(result != target) # 6 erreurs sur 150 mesures
print("Nb erreurs:", errors)
print( "Pourcentage de prédiction juste:", (150-errors)*100/150)   # 96 % de réussite


In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(result, target) # 96% de réussite




In [ ]:
from sklearn.metrics import confusion_matrix
conf = confusion_matrix(target, result)
conf


In [ ]:
sns.heatmap(conf, square=True, annot=True, cbar=False
            , xticklabels=list(iris.target_names)
            , yticklabels=list(iris.target_names))
plt.xlabel('valeurs prédites')
plt.ylabel('valeurs réelles');


## Séparation du jeu de tests et d'apprentissage
La fonction train_test_split permet de décomposer le jeu de données en 2 groupes: les données pour l'apprentissage et les données pour les tests.

Le paramètre train_size indique la taille du jeu d'apprentissage qui sera utilisé: 50% des enregistrements.



In [ ]:
# from sklearn.cross_validation import train_test_split # Version 0.17.1
from sklearn.model_selection import train_test_split # version 0.18.1
# split the data with 50% in each set
data_test = train_test_split(data, target
                                 , random_state=0
                                 , train_size=0.5)
data_train, data_test, target_train, target_test = data_test

In [ ]:
data_test[:5]

In [ ]:
#Il ne reste plus qu'à relancer la classification:
clf = GaussianNB()
clf.fit(data_train, target_train)
result = clf.predict(data_test)

In [ ]:
# Score
accuracy_score(result, target_test)

In [ ]:
# Matrice de confusion
conf = confusion_matrix(target_test, result)
conf


In [ ]:
sns.heatmap(conf, square=True, annot=True, cbar=False
            , xticklabels=list(iris.target_names)
            , yticklabels=list(iris.target_names))
plt.xlabel('valeurs prédites')
plt.ylabel('valeurs réelles');



## 2. KNN

In [ ]:
from sklearn import neighbors
clf = neighbors.KNeighborsClassifier()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

# Charger le dataset Iris et limiter à deux caractéristiques pour faciliter la visualisation
iris = datasets.load_iris()
X = iris.data[:, :2]  # Utiliser seulement les deux premières caractéristiques
y = iris.target

# Diviser les données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Créer et entraîner le classifieur KNN
clf = KNeighborsClassifier(n_neighbors=5)
clf.fit(X_train, y_train)  # S'assurer que le modèle est bien entraîné ici

# Définir la grille de points pour la visualisation
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))

# Prédire les classes pour chaque point de la grille
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Créer la figure et visualiser les déductions en couleurs
plt.figure(1)
plt.pcolormesh(xx, yy, Z, shading='auto', cmap=plt.cm.Pastel1)

# Afficher aussi les points d'entraînement
colors = ['violet', 'yellow', 'red']
C = [colors[i] for i in y]
plt.scatter(X[:, 0], X[:, 1], c=C, edgecolor='k', s=30)
plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())
plt.xlabel('Longueur du sépal (cm)')
plt.ylabel('Largeur du sépal (cm)')
plt.title("Surface de décision pour le classifieur KNN")
plt.show()



In [ ]:
data_test, target_test = iris.data[::2], iris.target[::2]
data_train, target_train = iris.data[1::2], iris.target[1::2]
result = []
n_values = range(1,20)
for n in n_values:
    clf = neighbors.KNeighborsClassifier(n_neighbors=n)
    clf.fit(data_train, target_train)
    Z = clf.predict(data_test)
    score = accuracy_score(Z, target_test)
    result.append(score)

plt.plot(list(n_values), result)

## 3. PCA

In [ ]:
# Réduction de dimension: analyse en composante principale.

from sklearn.decomposition import PCA  
# Définition de l'hyperparamètre du nombre de composantes voulues
model = PCA(n_components=2)
# Alimentation du modèle
model.fit(iris.data)
# Transformation avec ses propres données
reduc = model.transform(iris.data )


In [ ]:
# Données Originales
iris.data[:5]

In [ ]:
# Données réduites
reduc[:5]

In [ ]:
df.head()

In [ ]:
df['PCA1'] = reduc[:, 0]
df['PCA2'] = reduc[:, 1]
df.head()


In [ ]:
colors = ['violet', 'yellow', 'blue']
plt.scatter(df['PCA1'], df['PCA2'], c=[ colors[c] for c in df['target'] ]);
plt.xlabel('PCA1')
plt.ylabel('PCA2');

In [ ]:
sns.lmplot(x="PCA1", y="PCA2", hue='label', data=df, fit_reg=False);




Le résultat est plutôt satisfaisant, et ce qui est assez bluffant c'est que l'algorithme ne connaîssait pas du tout les types de fleurs !

## 4. Clustering: Gaussian Mixture Models (GMM)

In [ ]:
from sklearn.mixture import GaussianMixture  
# Création du modèle avec 3 groupes de données
model = GaussianMixture (n_components=3, covariance_type='full')  
# Apprentissage, il n'y en a pas vraiment
model.fit(df[['PCA1', 'PCA2']])   
# Prédiction
groups = model.predict(df[['PCA1', 'PCA2']])         


In [ ]:
df['group'] = groups
sns.lmplot(x="PCA1", y="PCA2", data=df, hue='label',
           col='group', fit_reg=False);      


Le groupe setosa est très bien identifié, il y a toujours une imprécision entre les classes virginica et versicolor mais le résultat reste remarquable sur ce petit échantillon.